# Impact of Futures and Options Trading on Volatility in the Indian Stock Market
### NIFTY 50 | End-to-end Python research notebook | 2021–2026

This notebook documents the complete workflow used in the project: market-data collection, NSE F&O archive collection, cleaning and classification, daily aggregation, merging, exploratory analysis, econometric regressions, forecasting, diagnostics and chart generation.

**How to run**
1. Run the setup and configuration cells.
2. Run the NIFTY 50 download cell.
3. Run the F&O collection cell. It downloads and checkpoints archives; this may take time.
4. Run the cleaning, aggregation, merge and analysis sections in order.
5. The `data/` and `visualizations/` folders are created automatically.

The F&O downloader uses a public GitHub data repository. Check that repository's data provenance and terms before redistribution. If you already have the consolidated F&O CSV, you may skip the download cell and place it at `data/nifty_fno_2021_2026.csv`.

**Important:** This is an observational research exercise. Association does not establish causation. Forecasting target and historical-volatility regression target are different and are identified separately below.

## 1. Environment and project configuration

In [ ]:
%pip install -q yfinance pandas numpy matplotlib requests statsmodels scikit-learn urllib3

In [ ]:
from pathlib import Path
import os
import io
import time
import hashlib
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf
import statsmodels.api as sm

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

# All project files are stored relative to this notebook.
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
GRAPH_DIR = BASE_DIR / "visualizations"
CHECKPOINT_DIR = DATA_DIR / "fno_checkpoints"

for folder in [DATA_DIR, GRAPH_DIR, CHECKPOINT_DIR / "parts"]:
    folder.mkdir(parents=True, exist_ok=True)

START_DATE = pd.Timestamp("2021-01-01")
END_DATE = pd.Timestamp("2026-10-24")  # Yahoo/NSE data through 23 Oct 2026 in this project

NIFTY_FILE = DATA_DIR / "nifty50_cleaned.csv"
FNO_RAW_FILE = DATA_DIR / "nifty_fno_2021_2026.csv"
FNO_FIXED_FILE = DATA_DIR / "nifty_fno_2021_2026_fixed.csv"
FNO_DAILY_FILE = DATA_DIR / "fno_daily_aggregated_fixed.csv"
MERGED_FILE = DATA_DIR / "nifty_fno_merged_daily_fixed.csv"

print("Project directory:", BASE_DIR)
print("Data directory:", DATA_DIR.resolve())
print("Visualizations directory:", GRAPH_DIR.resolve())

## 2. Download and prepare NIFTY 50 historical prices

In [ ]:
# NIFTY 50 ticker on Yahoo Finance
nifty_raw = yf.download(
    "^NSEI",
    start=START_DATE.strftime("%Y-%m-%d"),
    end=END_DATE.strftime("%Y-%m-%d"),
    auto_adjust=True,
    progress=False
)

if nifty_raw.empty:
    raise ValueError("No NIFTY 50 data received. Check internet access or the ticker.")

# yfinance may return MultiIndex columns depending on its version.
if isinstance(nifty_raw.columns, pd.MultiIndex):
    nifty_raw.columns = nifty_raw.columns.get_level_values(0)

nifty_raw = nifty_raw.sort_index()
nifty_raw = nifty_raw[~nifty_raw.index.duplicated(keep="first")]

nifty = nifty_raw.reset_index()
nifty = nifty.rename(columns={nifty.columns[0]: "Date"})
nifty["Date"] = pd.to_datetime(nifty["Date"], errors="coerce").dt.normalize()
nifty = nifty.dropna(subset=["Date"]).copy()

# Standardise price fields and calculate returns / rolling volatility.
nifty["Close"] = pd.to_numeric(nifty["Close"], errors="coerce")
nifty["Daily_Return_Pct"] = nifty["Close"].pct_change() * 100
nifty["Volatility_20d_Pct"] = (
    nifty["Daily_Return_Pct"].rolling(20).std() * np.sqrt(252)
)

nifty = nifty.dropna(subset=["Date", "Close"]).sort_values("Date")
nifty.to_csv(NIFTY_FILE, index=False)

print("Saved:", NIFTY_FILE.resolve())
print("Shape:", nifty.shape)
print("Date range:", nifty["Date"].min(), "to", nifty["Date"].max())
display(nifty.head())

### NIFTY 50 price and volatility preview

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(nifty["Date"], nifty["Close"], linewidth=1.3)
axes[0].set_title("NIFTY 50 Historical Closing Price")
axes[0].set_ylabel("Index level")
axes[0].grid(alpha=0.25)

axes[1].plot(nifty["Date"], nifty["Volatility_20d_Pct"], linewidth=1.1)
axes[1].set_title("Annualised 20-Day Rolling Volatility")
axes[1].set_ylabel("Volatility (%)")
axes[1].set_xlabel("Date")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Collect NSE F&O contract-level data

The following cell is based on the checkpointed archive-collection workflow used in the project. It:
- obtains the archive listing from the public NSE F&O data-bank repository;
- downloads ZIP archives one at a time;
- standardises legacy and newer NSE column names;
- saves each processed archive as a checkpoint so interrupted runs can resume;
- writes a consolidated raw CSV.

**Runtime note:** This is the longest stage and depends on GitHub availability. If `nifty_fno_2021_2026.csv` already exists in `data/`, you can skip this cell.

In [ ]:
import io
import time
import hashlib
import zipfile
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# 1. SETTINGS
# ============================================================

REPO_API = (
    "https://api.github.com/repos/"
    "SantoshSrinivas79/NSE-FNO-Data-bank/git/trees/main?recursive=1"
)

RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "SantoshSrinivas79/NSE-FNO-Data-bank/main/"
)

START_DATE = pd.Timestamp("2021-01-01")
END_DATE = pd.Timestamp("2026-10-24")

OUTPUT = FNO_RAW_FILE
CHECKPOINT_DIR = DATA_DIR / "fno_checkpoints"
PARTS_DIR = CHECKPOINT_DIR / "parts"
PROCESSED_LOG = CHECKPOINT_DIR / "processed_paths.txt"
FAILURE_LOG = DATA_DIR / "nifty_fno_download_failures.csv"

CHECKPOINT_DIR.mkdir(exist_ok=True)
PARTS_DIR.mkdir(exist_ok=True)

STANDARD_COLUMNS = [
    "trade_date",
    "symbol",
    "expiry",
    "strike",
    "option_type",
    "open",
    "high",
    "low",
    "close",
    "settlement",
    "volume_contracts",
    "turnover_source_units",
    "open_interest",
    "change_in_oi",
    "contract_type",
    "source_file",
]


# ============================================================
# 2. HTTP SESSION WITH RETRIES
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "NIFTY-FNO-Research-Project"
})

retry_config = Retry(
    total=4,
    connect=4,
    read=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

adapter = HTTPAdapter(max_retries=retry_config)

session.mount("https://", adapter)


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def find_col(df, *names):
    """Find a column without depending on capitalization."""
    lookup = {
        str(c).strip().upper(): c
        for c in df.columns
    }

    for name in names:
        if name.strip().upper() in lookup:
            return lookup[name.strip().upper()]

    return None


def get_series(df, *names):
    """Return a column or an NA-filled Series with matching index."""
    col = find_col(df, *names)

    if col is None:
        return pd.Series(
            pd.NA,
            index=df.index,
            dtype="object"
        )

    return df[col]


def parse_trade_dates(values):
    """Parse common NSE date formats without ambiguous Boolean masks."""
    s = values.astype("string").str.strip()

    result = pd.Series(
        pd.NaT,
        index=s.index,
        dtype="datetime64[ns]"
    )

    # Format such as 20240715
    mask_yyyymmdd = s.str.fullmatch(
        r"\d{8}", na=False
    )

    result.loc[mask_yyyymmdd] = pd.to_datetime(
        s.loc[mask_yyyymmdd],
        format="%Y%m%d",
        errors="coerce"
    )

    # Format such as 2024-07-15
    mask_iso = s.str.fullmatch(
        r"\d{4}-\d{2}-\d{2}", na=False
    )

    result.loc[mask_iso] = pd.to_datetime(
        s.loc[mask_iso],
        format="%Y-%m-%d",
        errors="coerce"
    )

    # Remaining formats, including common legacy NSE dates
    mask_other = ~(mask_yyyymmdd | mask_iso)

    result.loc[mask_other] = pd.to_datetime(
        s.loc[mask_other],
        errors="coerce",
        dayfirst=True
    )

    return result


def part_path(path):
    """Create a stable checkpoint filename for an archive."""
    digest = hashlib.sha1(
        path.encode("utf-8")
    ).hexdigest()

    return PARTS_DIR / f"{digest}.csv"


def load_processed_paths():
    if not PROCESSED_LOG.exists():
        return set()

    return {
        line.strip()
        for line in PROCESSED_LOG.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    }


processed_paths = load_processed_paths()


def mark_processed(path, df):
    """Save an archive's results before marking it complete."""
    destination = part_path(path)

    df.to_csv(destination, index=False)

    if path not in processed_paths:
        with PROCESSED_LOG.open(
            "a", encoding="utf-8"
        ) as f:
            f.write(path + "\n")

        processed_paths.add(path)


def save_failures(failures):
    pd.DataFrame(
        failures,
        columns=["file", "error"]
    ).to_csv(FAILURE_LOG, index=False)


# ============================================================
# 4. PRESERVE DATA FROM THE PREVIOUS NOTEBOOK RUN
# ============================================================

# If the previous run's frames still exist in this kernel,
# migrate them into checkpoints so they don't need downloading again.

old_frames = globals().get("frames", [])

if isinstance(old_frames, list) and old_frames:
    migrated = 0

    for df in old_frames:
        if not isinstance(df, pd.DataFrame):
            continue

        if "source_file" not in df.columns:
            continue

        if df.empty:
            continue

        for source_path, group in df.groupby(
            "source_file", dropna=True
        ):
            if pd.isna(source_path):
                continue

            source_path = str(source_path)

            if source_path not in processed_paths:
                saved = group.reindex(
                    columns=STANDARD_COLUMNS
                ).copy()

                mark_processed(source_path, saved)
                migrated += 1

    print(
        "Previous successful archive checkpoints migrated:",
        migrated
    )

else:
    print(
        "No previous in-memory frames found. "
        "Existing disk checkpoints will still be reused."
    )


# ============================================================
# 5. GET THE ARCHIVE LIST
# ============================================================

response = session.get(REPO_API, timeout=90)
response.raise_for_status()

tree_data = response.json()

if tree_data.get("truncated", False):
    raise RuntimeError(
        "GitHub returned a truncated file listing. "
        "Do not start downloading until the archive list is complete."
    )

tree = tree_data.get("tree", [])

zip_paths = []

for item in tree:
    path = item.get("path", "")
    pieces = path.split("/")

    if (
        path.startswith("data/")
        and path.endswith(".zip")
        and len(pieces) >= 4
        and pieces[1].isdigit()
        and 2021 <= int(pieces[1]) <= 2026
    ):
        zip_paths.append(path)

zip_paths.sort()

print("Daily archives found:", len(zip_paths))
print("Already completed:", len(processed_paths))
print("Archives to check:", len(zip_paths) - sum(
    p in processed_paths for p in zip_paths
))


# ============================================================
# 6. PROCESS ARCHIVES ONE BY ONE
# ============================================================

failures = []

for i, path in enumerate(zip_paths, start=1):

    # Skip archives already saved successfully
    if path in processed_paths:
        continue

    url = RAW_BASE + path

    try:
        # Download archive
        r = session.get(url, timeout=120)
        r.raise_for_status()

        # Extract the first CSV
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            csv_names = [
                name for name in z.namelist()
                if name.lower().endswith(".csv")
            ]

            if not csv_names:
                raise ValueError("No CSV file inside ZIP")

            with z.open(csv_names[0]) as f:
                raw = pd.read_csv(
                    f,
                    low_memory=False
                )

        # ----------------------------------------------------
        # CRITICAL FIX: MISSING SYMBOLS MUST NOT CREATE
        # MISSING BOOLEAN VALUES IN THE ROW FILTER
        # ----------------------------------------------------

        symbol = get_series(
            raw, "SYMBOL", "TckrSymb"
        )

        mask = (
            symbol.astype("string")
            .str.strip()
            .str.upper()
            .eq("NIFTY")
            .fillna(False)
        )

        nifty = raw.loc[mask].copy()

        # Save a valid empty checkpoint if no NIFTY rows exist
        if nifty.empty:
            empty = pd.DataFrame(columns=STANDARD_COLUMNS)
            mark_processed(path, empty)

            if i % 25 == 0:
                print(
                    f"Checked {i}/{len(zip_paths)} archives"
                )

            time.sleep(0.15)
            continue

        # ----------------------------------------------------
        # STANDARDISE OLD AND NEW NSE COLUMN NAMES
        # ----------------------------------------------------

        out = pd.DataFrame(index=nifty.index)

        out["trade_date"] = get_series(
            nifty, "TIMESTAMP", "TradDt", "BizDt"
        )

        out["symbol"] = get_series(
            nifty, "SYMBOL", "TckrSymb"
        )

        out["expiry"] = get_series(
            nifty,
            "EXPIRY_DT",
            "XpryDt",
            "FininstrmActlXpryDt"
        )

        out["strike"] = get_series(
            nifty, "STRIKE_PR", "StrkPric"
        )

        out["option_type"] = get_series(
            nifty, "OPTION_TYP", "OptnTp"
        )

        out["open"] = get_series(
            nifty, "OPEN", "OpnPric"
        )

        out["high"] = get_series(
            nifty, "HIGH", "HghPric"
        )

        out["low"] = get_series(
            nifty, "LOW", "LwPric"
        )

        out["close"] = get_series(
            nifty, "CLOSE", "ClsPric"
        )

        out["settlement"] = get_series(
            nifty, "SETTLE_PR", "SttlmPric"
        )

        out["volume_contracts"] = get_series(
            nifty, "CONTRACTS", "TtlTradgVol"
        )

        out["turnover_source_units"] = get_series(
            nifty, "VAL_INLAKH", "TtlTrfVal"
        )

        out["open_interest"] = get_series(
            nifty, "OPEN_INT", "OpnIntrst"
        )

        out["change_in_oi"] = get_series(
            nifty, "CHG_IN_OI", "ChngInOpnIntrst"
        )

        # ----------------------------------------------------
        # PARSE DATES AND FILTER THE REQUESTED PERIOD
        # ----------------------------------------------------

        out["trade_date"] = parse_trade_dates(
            out["trade_date"]
        )

        date_mask = (
            out["trade_date"]
            .between(START_DATE, END_DATE)
            .fillna(False)
        )

        out = out.loc[date_mask].copy()

        # ----------------------------------------------------
        # CRITICAL FIX: CLASSIFY CONTRACTS VECTORIZED
        # ----------------------------------------------------

        opt = (
            out["option_type"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

        out["contract_type"] = pd.Series(
            pd.NA,
            index=out.index,
            dtype="string"
        )

        # CE and PE identify option rows
        option_mask = opt.isin(["CE", "PE"])
        out.loc[option_mask, "contract_type"] = "OPTIDX"

        # Use explicit instrument information where available
        instrument = get_series(
            nifty,
            "INSTRUMENT",
            "FinInstrmTp"
        ).reindex(out.index)

        instrument = (
            instrument.astype("string")
            .str.strip()
            .str.upper()
        )

        valid_instrument = instrument.isin(
            ["FUTIDX", "OPTIDX"]
        )

        out.loc[
            valid_instrument,
            "contract_type"
        ] = instrument.loc[valid_instrument]

        # Do NOT automatically label every unknown row as a future.

        # ----------------------------------------------------
        # NUMERIC CONVERSION
        # ----------------------------------------------------

        numeric_cols = [
            "strike",
            "open",
            "high",
            "low",
            "close",
            "settlement",
            "volume_contracts",
            "turnover_source_units",
            "open_interest",
            "change_in_oi",
        ]

        for col in numeric_cols:
            out[col] = pd.to_numeric(
                out[col],
                errors="coerce"
            )

        out["source_file"] = path

        out = out.reindex(
            columns=STANDARD_COLUMNS
        ).reset_index(drop=True)

        # Save each archive immediately for safe resumption
        mark_processed(path, out)

        if i % 25 == 0:
            print(
                f"Processed {i}/{len(zip_paths)} archives; "
                f"checkpointed archives: {len(processed_paths)}"
            )

        time.sleep(0.15)

    except Exception as e:
        failures.append((path, repr(e)))
        save_failures(failures)

        print(f"Failed: {path} — {repr(e)}")

# ============================================================
# 7. BUILD THE CONSOLIDATED CSV FROM CHECKPOINTS
# ============================================================

parts = []

for path in zip_paths:
    if path not in processed_paths:
        continue

    saved_part = part_path(path)

    if not saved_part.exists():
        continue

    part = pd.read_csv(
        saved_part,
        low_memory=False
    )

    if not part.empty:
        parts.append(part)

if not parts:
    raise RuntimeError(
        "No non-empty checkpoint data found. "
        "Inspect the failure log before continuing."
    )

combined = pd.concat(
    parts,
    ignore_index=True
)

combined["trade_date"] = pd.to_datetime(
    combined["trade_date"],
    errors="coerce"
)

combined = combined.loc[
    combined["trade_date"]
    .between(START_DATE, END_DATE)
    .fillna(False)
].copy()

combined = combined.sort_values(
    [
        "trade_date",
        "contract_type",
        "expiry",
        "strike",
        "option_type"
    ],
    na_position="last"
)

combined.to_csv(
    OUTPUT,
    index=False
)

# ============================================================
# 8. SUMMARY AND RETRY LOG
# ============================================================

print("\n========== PROCESSING SUMMARY ==========")
print("Output file:", OUTPUT.resolve())
print("Rows:", f"{len(combined):,}")
print(
    "Date range:",
    combined["trade_date"].min(),
    "to",
    combined["trade_date"].max()
)

print("\nContract types:")
print(
    combined["contract_type"]
    .value_counts(dropna=False)
)

unknown_count = combined["contract_type"].isna().sum()

print(
    "\nRows needing contract-type verification:",
    f"{unknown_count:,}"
)

print(
    "Successfully checkpointed archives:",
    len(processed_paths)
)

print("Failed archives this run:", len(failures))

if failures:
    print("Failure log:", FAILURE_LOG.resolve())

print("========================================")

## 4. Inspect the downloaded contract-level dataset

In [ ]:
if not FNO_RAW_FILE.exists():
    raise FileNotFoundError(
        f"Missing {FNO_RAW_FILE}. Run the F&O downloader or place the consolidated CSV in data/."
    )

fno_sample = pd.read_csv(FNO_RAW_FILE, low_memory=False)
print("Raw F&O shape:", fno_sample.shape)
print("Date range:", fno_sample["trade_date"].min(), "to", fno_sample["trade_date"].max())
print("\nContract type counts:")
display(fno_sample["contract_type"].value_counts(dropna=False).to_frame("rows"))
print("\nOption type counts:")
display(fno_sample["option_type"].value_counts(dropna=False).to_frame("rows"))
display(fno_sample.head())
del fno_sample

## 5. Validate contract classification

The source notebook used explicit `CE` / `PE` option markers and the `XX` marker for futures. In the NIFTY-only dataset, rows with missing option type and missing strike were also treated as futures in the correction pass. Other ambiguous rows are retained as unclassified rather than silently assigned.

In [ ]:
chunksize = 200_000
first_chunk = True

for chunk in pd.read_csv(FNO_RAW_FILE, chunksize=chunksize, low_memory=False):
    option_type = chunk["option_type"].astype("string").str.strip().str.upper()
    strike = pd.to_numeric(chunk["strike"], errors="coerce")

    contract_type = pd.Series(pd.NA, index=chunk.index, dtype="string")
    contract_type.loc[option_type.isin(["CE", "PE"])] = "OPTIDX"
    contract_type.loc[option_type.eq("XX")] = "FUTIDX"

    # Project-specific inference used for this NIFTY-only archive.
    inferred_futures = option_type.isna() & strike.isna()
    contract_type.loc[inferred_futures] = "FUTIDX"

    chunk["contract_type"] = contract_type
    chunk.to_csv(
        FNO_FIXED_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )
    first_chunk = False

print("Saved corrected file:", FNO_FIXED_FILE.resolve())

In [ ]:
# Verify the classification after the chunked correction.
year_counts = None
unknown_rows = 0

for chunk in pd.read_csv(FNO_FIXED_FILE, chunksize=chunksize, low_memory=False):
    dates = pd.to_datetime(chunk["trade_date"], errors="coerce")
    temp = pd.DataFrame({
        "year": dates.dt.year,
        "contract_type": chunk["contract_type"]
    }).dropna(subset=["year"])
    unknown_rows += (~temp["contract_type"].isin(["FUTIDX", "OPTIDX"])).sum()
    counts = pd.crosstab(temp["year"], temp["contract_type"])
    year_counts = counts if year_counts is None else year_counts.add(counts, fill_value=0)

display(year_counts.fillna(0).astype(int))
print("Rows still needing contract-type verification:", unknown_rows)

## 6. Aggregate contract-level F&O records to daily indicators

In [ ]:
chunksize = 200_000
daily_parts = []
call_put_parts = []

for chunk in pd.read_csv(FNO_FIXED_FILE, chunksize=chunksize, low_memory=False):
    chunk["trade_date"] = pd.to_datetime(
        chunk["trade_date"], errors="coerce"
    ).dt.normalize()

    for col in ["volume_contracts", "open_interest", "change_in_oi"]:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk["option_type"] = (
        chunk["option_type"].astype("string").str.strip().str.upper()
    )
    chunk = chunk.dropna(subset=["trade_date", "contract_type"])

    for col in ["volume_contracts", "open_interest", "change_in_oi"]:
        chunk[col] = chunk[col].fillna(0)

    is_fut = chunk["contract_type"].eq("FUTIDX")
    is_opt = chunk["contract_type"].eq("OPTIDX")
    is_call = is_opt & chunk["option_type"].eq("CE")
    is_put = is_opt & chunk["option_type"].eq("PE")

    chunk["futures_volume"] = chunk["volume_contracts"].where(is_fut, 0)
    chunk["futures_open_interest"] = chunk["open_interest"].where(is_fut, 0)
    chunk["futures_change_in_oi"] = chunk["change_in_oi"].where(is_fut, 0)

    chunk["options_volume"] = chunk["volume_contracts"].where(is_opt, 0)
    chunk["options_open_interest"] = chunk["open_interest"].where(is_opt, 0)
    chunk["options_change_in_oi"] = chunk["change_in_oi"].where(is_opt, 0)

    daily_cols = [
        "futures_volume", "futures_open_interest", "futures_change_in_oi",
        "options_volume", "options_open_interest", "options_change_in_oi"
    ]
    daily_parts.append(chunk.groupby("trade_date")[daily_cols].sum())

    cp = chunk[is_call | is_put].groupby(
        ["trade_date", "option_type"]
    ).agg(
        volume=("volume_contracts", "sum"),
        open_interest=("open_interest", "sum")
    )
    if not cp.empty:
        call_put_parts.append(cp)

fno_daily = pd.concat(daily_parts).groupby(level=0).sum().sort_index()

if call_put_parts:
    cp_daily = pd.concat(call_put_parts).groupby(level=[0, 1]).sum().unstack("option_type", fill_value=0)
else:
    cp_daily = pd.DataFrame(index=fno_daily.index)

for metric in ["volume", "open_interest"]:
    for side, label in [("CE", "call"), ("PE", "put")]:
        key = (metric, side)
        col_name = f"{label}_{metric}"
        if key in cp_daily.columns:
            fno_daily[col_name] = cp_daily[key].reindex(fno_daily.index).fillna(0)
        else:
            fno_daily[col_name] = 0

fno_daily["put_call_volume_ratio"] = (
    fno_daily["put_volume"] / fno_daily["call_volume"].replace(0, np.nan)
)
fno_daily["put_call_oi_ratio"] = (
    fno_daily["put_open_interest"] /
    fno_daily["call_open_interest"].replace(0, np.nan)
)

fno_daily.index.name = "Date"
fno_daily.to_csv(FNO_DAILY_FILE)
print("Saved daily F&O indicators:", FNO_DAILY_FILE.resolve())
print("Daily rows:", len(fno_daily))
display(fno_daily.head())

## 7. Merge NIFTY 50 and F&O daily datasets

In [ ]:
nifty = pd.read_csv(NIFTY_FILE, parse_dates=["Date"])
fno = pd.read_csv(FNO_DAILY_FILE, parse_dates=["Date"])

nifty["Date"] = pd.to_datetime(nifty["Date"], errors="coerce").dt.normalize()
fno["Date"] = pd.to_datetime(fno["Date"], errors="coerce").dt.normalize()
nifty = nifty.dropna(subset=["Date"])
fno = fno.dropna(subset=["Date"])

assert not nifty["Date"].duplicated().any(), "NIFTY dataset has duplicate dates."
assert not fno["Date"].duplicated().any(), "F&O dataset has duplicate dates."

merged = nifty.merge(
    fno, on="Date", how="left", indicator=True, validate="one_to_one"
)
merged["fno_data_available"] = merged["_merge"].eq("both")
missing_dates = merged.loc[~merged["fno_data_available"], "Date"]
merged = merged.drop(columns="_merge").sort_values("Date").reset_index(drop=True)

merged.to_csv(MERGED_FILE, index=False)
print("Saved merged dataset:", MERGED_FILE.resolve())
print("Shape:", merged.shape)
print("NIFTY dates without matched F&O data:", len(missing_dates))
display(merged.head())

## 8. Exploratory data analysis (EDA)

In [ ]:
df = pd.read_csv(MERGED_FILE, parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)

analysis_cols = [
    "Daily_Return_Pct", "Volatility_20d_Pct",
    "futures_volume", "futures_open_interest", "futures_change_in_oi",
    "options_volume", "options_open_interest", "options_change_in_oi",
    "call_volume", "put_volume", "call_open_interest", "put_open_interest",
    "put_call_volume_ratio", "put_call_oi_ratio"
]
analysis_cols = [c for c in analysis_cols if c in df.columns]

for col in analysis_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Dataset shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("\nMissing values:")
display(df[analysis_cols].isna().sum().to_frame("missing"))
print("\nDescriptive statistics:")
display(df[analysis_cols].describe().T.round(4))

matched = df[df["fno_data_available"].fillna(False)].copy()
corr_cols = [c for c in analysis_cols if c in matched.columns]
spearman_corr = matched[corr_cols].corr(method="spearman")
print("\nSpearman correlation matrix:")
display(spearman_corr.round(3))

## 9. Visualisations

In [ ]:
# Graph 1: NIFTY 50 closing price
plot_df = df.dropna(subset=["Date", "Close"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df["Date"], plot_df["Close"], linewidth=1.4)
ax.set(title="NIFTY 50 Closing Price (2021–2026)", xlabel="Date", ylabel="Closing index level")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(GRAPH_DIR / "01_nifty50_price.png", dpi=250)
plt.show()

# Graph 2: 20-day rolling volatility
plot_df = df.dropna(subset=["Date", "Volatility_20d_Pct"])
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df["Date"], plot_df["Volatility_20d_Pct"], linewidth=1.2)
ax.set(title="NIFTY 50: 20-Day Rolling Volatility", xlabel="Date", ylabel="20-Day volatility (%)")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(GRAPH_DIR / "02_nifty50_volatility.png", dpi=250)
plt.show()

# Graphs 3 and 4: futures/options volume against volatility
for volume_col, title, filename, xlabel in [
    ("futures_volume", "Futures Volume vs 20-Day Volatility", "03_futures_volume_vs_volatility.png", "Log(1 + futures volume)"),
    ("options_volume", "Options Volume vs 20-Day Volatility", "04_options_volume_vs_volatility.png", "Log(1 + options volume)")
]:
    plot_df = matched.dropna(subset=[volume_col, "Volatility_20d_Pct"]).copy()
    plot_df = plot_df[plot_df[volume_col] >= 0]
    plot_df["log_volume"] = np.log1p(plot_df[volume_col])
    rho = plot_df[volume_col].corr(plot_df["Volatility_20d_Pct"], method="spearman")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(plot_df["log_volume"], plot_df["Volatility_20d_Pct"], alpha=0.4, s=16)
    x = plot_df["log_volume"].to_numpy()
    y = plot_df["Volatility_20d_Pct"].to_numpy()
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() > 1:
        slope, intercept = np.polyfit(x[valid], y[valid], 1)
        x_line = np.linspace(x[valid].min(), x[valid].max(), 100)
        ax.plot(x_line, slope*x_line + intercept, linewidth=2)
    ax.set(title=f"{title}\nSpearman correlation = {rho:.3f}", xlabel=xlabel, ylabel="20-Day volatility (%)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(GRAPH_DIR / filename, dpi=250)
    plt.show()

# Graph 5: Spearman correlation heatmap (log activity measures)
heatmap_cols = [
    "Volatility_20d_Pct", "Daily_Return_Pct", "futures_volume",
    "options_volume", "futures_open_interest", "options_open_interest"
]
corr_df = matched[heatmap_cols].copy()
for col in heatmap_cols:
    if col in ["futures_volume", "options_volume", "futures_open_interest", "options_open_interest"]:
        corr_df[col] = np.log1p(corr_df[col].clip(lower=0))
corr = corr_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
fig.colorbar(im, ax=ax, label="Spearman correlation")
ax.set_xticks(range(len(corr.columns))); ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=40, ha="right")
ax.set_yticklabels(corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Correlation Between Market and F&O Variables")
fig.tight_layout()
fig.savefig(GRAPH_DIR / "05_correlation_heatmap.png", dpi=250)
plt.show()

## 10. Model A — main OLS regression

**Dependent variable:** 20-day rolling volatility (%).

Explanatory variables include daily return, log futures/options volume, log futures/options open interest and year indicators. The model uses HAC standard errors with five lags.

In [ ]:
model_df = df.copy()
model_cols = [
    "Volatility_20d_Pct", "Daily_Return_Pct", "futures_volume",
    "options_volume", "futures_open_interest", "options_open_interest"
]
for col in model_cols:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

for col in ["futures_volume", "options_volume", "futures_open_interest", "options_open_interest"]:
    model_df["log_" + col] = np.log1p(model_df[col].clip(lower=0))

model_df["year"] = model_df["Date"].dt.year
required = [
    "Volatility_20d_Pct", "Daily_Return_Pct", "log_futures_volume",
    "log_options_volume", "log_futures_open_interest",
    "log_options_open_interest", "year", "fno_data_available"
]
regression_data = (
    model_df[required].replace([np.inf, -np.inf], np.nan).dropna()
)
regression_data = regression_data[regression_data["fno_data_available"].astype(bool)].copy()

year_dummies = pd.get_dummies(
    regression_data["year"].astype(int), prefix="year", drop_first=True, dtype=float
)
X_a = regression_data[[
    "Daily_Return_Pct", "log_futures_volume", "log_options_volume",
    "log_futures_open_interest", "log_options_open_interest"
]].astype(float).reset_index(drop=True)
X_a = pd.concat([X_a, year_dummies.reset_index(drop=True)], axis=1)
X_a = sm.add_constant(X_a, has_constant="add")
y_a = regression_data["Volatility_20d_Pct"].astype(float).reset_index(drop=True)

model_a = sm.OLS(y_a, X_a).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(model_a.summary())
display(model_a.params.rename("Coefficient").to_frame().join(
    model_a.bse.rename("Std. Error")
).join(model_a.pvalues.rename("p-value")).round(5))

### Model A diagnostics: predictor correlation and VIF

In [ ]:
X_vif = X_a.astype(float)
vif_table = pd.DataFrame({
    "Variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
display(vif_table.round(3))
print("Durbin-Watson:", round(sm.stats.stattools.durbin_watson(model_a.resid), 3))

## 11. Model B — lagged-volatility regression

Model B adds the previous observed trading day's 20-day rolling volatility. This specification tests volatility persistence. Its high R-squared must not be interpreted as the proportion of volatility explained by F&O activity alone.

In [ ]:
model_b_df = df.copy()
for col in ["Volatility_20d_Pct", "Daily_Return_Pct", "futures_volume", "options_volume"]:
    model_b_df[col] = pd.to_numeric(model_b_df[col], errors="coerce")

model_b_df["lag_volatility"] = model_b_df["Volatility_20d_Pct"].shift(1)
model_b_df["log_futures_volume"] = np.log1p(model_b_df["futures_volume"].clip(lower=0))
model_b_df["log_options_volume"] = np.log1p(model_b_df["options_volume"].clip(lower=0))
model_b_df["year"] = model_b_df["Date"].dt.year

features_b = [
    "lag_volatility", "Daily_Return_Pct",
    "log_futures_volume", "log_options_volume"
]
required_b = ["Date", "Volatility_20d_Pct", "fno_data_available", "year"] + features_b
data_b = model_b_df[required_b].replace([np.inf, -np.inf], np.nan).dropna()
data_b = data_b[data_b["fno_data_available"].astype(bool)].copy()

year_dummies_b = pd.get_dummies(
    data_b["year"].astype(int), prefix="year", drop_first=True, dtype=float
)
X_b = pd.concat([
    data_b[features_b].astype(float).reset_index(drop=True),
    year_dummies_b.reset_index(drop=True)
], axis=1)
X_b = sm.add_constant(X_b, has_constant="add")
y_b = data_b["Volatility_20d_Pct"].astype(float).reset_index(drop=True)

model_b = sm.OLS(y_b, X_b).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
print(model_b.summary())
display(model_b.params.rename("Coefficient").to_frame().join(
    model_b.bse.rename("Std. Error")
).join(model_b.pvalues.rename("p-value")).round(5))
print("Durbin-Watson:", round(sm.stats.stattools.durbin_watson(model_b.resid), 3))

## 12. Forecasting — next-day absolute return

**Forecasting target:** absolute daily return on the next observed trading day. This is not the same dependent variable as the 20-day rolling volatility used in Models A and B.

The split is chronological (first 80% for training, final 20% for testing) to avoid random shuffling and look-ahead leakage. Model features:
- Recent-volatility baseline: current 20-day rolling volatility.
- Market-only: current 20-day rolling volatility and today's absolute return.
- Market + F&O: market-only features plus log futures and options volume.

In [ ]:
forecast_df = df.copy()
for col in ["Daily_Return_Pct", "Volatility_20d_Pct", "futures_volume", "options_volume"]:
    forecast_df[col] = pd.to_numeric(forecast_df[col], errors="coerce")

forecast_df["target_next_day_abs_return"] = forecast_df["Daily_Return_Pct"].abs().shift(-1)
forecast_df["today_abs_return"] = forecast_df["Daily_Return_Pct"].abs()
forecast_df["log_futures_volume"] = np.log1p(forecast_df["futures_volume"].clip(lower=0))
forecast_df["log_options_volume"] = np.log1p(forecast_df["options_volume"].clip(lower=0))

required_forecast = [
    "target_next_day_abs_return", "Volatility_20d_Pct",
    "today_abs_return", "log_futures_volume", "log_options_volume",
    "fno_data_available"
]
forecast_data = forecast_df.replace([np.inf, -np.inf], np.nan).dropna(
    subset=required_forecast
).copy()
forecast_data = forecast_data[forecast_data["fno_data_available"].astype(bool)]
forecast_data = forecast_data.sort_values("Date").reset_index(drop=True)

split_index = int(len(forecast_data) * 0.80)
train = forecast_data.iloc[:split_index].copy()
test = forecast_data.iloc[split_index:].copy()

print("Training observations:", len(train))
print("Testing observations:", len(test))
print("Training dates:", train["Date"].min(), "to", train["Date"].max())
print("Testing dates:", test["Date"].min(), "to", test["Date"].max())

feature_sets = {
    "Recent-volatility baseline": ["Volatility_20d_Pct"],
    "Market-only": ["Volatility_20d_Pct", "today_abs_return"],
    "Market + F&O": [
        "Volatility_20d_Pct", "today_abs_return",
        "log_futures_volume", "log_options_volume"
    ]
}

y_train = train["target_next_day_abs_return"].astype(float)
y_test = test["target_next_day_abs_return"].astype(float)

predictions = {"Training-mean benchmark": np.full(len(test), y_train.mean())}
fitted_forecast_models = {}

for name, features in feature_sets.items():
    X_train = sm.add_constant(train[features].astype(float), has_constant="add")
    X_test = sm.add_constant(test[features].astype(float), has_constant="add")
    fitted = sm.OLS(y_train, X_train).fit()
    fitted_forecast_models[name] = fitted
    predictions[name] = fitted.predict(X_test).to_numpy()

overall_rows = []
for name, pred in predictions.items():
    overall_rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred)
    })

overall_results = pd.DataFrame(overall_rows).sort_values("MAE").reset_index(drop=True)
display(overall_results.round(5))
overall_results.to_csv(DATA_DIR / "forecast_performance_overall.csv", index=False)

### Forecast error distribution and largest misses

In [ ]:
comparison = pd.DataFrame({
    "Date": pd.to_datetime(test["Date"]).to_numpy(),
    "Actual": y_test.to_numpy(),
    "Baseline_Prediction": predictions["Recent-volatility baseline"],
    "Market_FNO_Prediction": predictions["Market + F&O"]
})
comparison["Baseline_Abs_Error"] = (
    comparison["Actual"] - comparison["Baseline_Prediction"]
).abs()
comparison["Market_FNO_Abs_Error"] = (
    comparison["Actual"] - comparison["Market_FNO_Prediction"]
).abs()
comparison["Baseline_Squared_Error"] = (
    comparison["Actual"] - comparison["Baseline_Prediction"]
) ** 2
comparison["Market_FNO_Squared_Error"] = (
    comparison["Actual"] - comparison["Market_FNO_Prediction"]
) ** 2

percentile_table = comparison[
    ["Baseline_Abs_Error", "Market_FNO_Abs_Error"]
].quantile([0.50, 0.90, 0.95, 0.99])
display(percentile_table.round(5))
percentile_table.to_csv(DATA_DIR / "forecast_error_percentiles.csv")

print("10 largest Market + F&O absolute errors:")
display(comparison.nlargest(10, "Market_FNO_Abs_Error").round(5))

top10 = comparison.nlargest(10, "Market_FNO_Squared_Error")
top10_share = top10["Market_FNO_Squared_Error"].sum() / comparison["Market_FNO_Squared_Error"].sum()
print(f"Share of total Market + F&O squared error from 10 largest misses: {top10_share:.2%}")

### Forecast performance by year

In [ ]:
year_rows = []
for name, pred in predictions.items():
    temp = pd.DataFrame({
        "Date": pd.to_datetime(test["Date"]).to_numpy(),
        "Actual": y_test.to_numpy(),
        "Predicted": pred
    })
    temp["Year"] = temp["Date"].dt.year
    temp["Absolute_Error"] = (temp["Actual"] - temp["Predicted"]).abs()
    temp["Squared_Error"] = (temp["Actual"] - temp["Predicted"]) ** 2

    for year, group in temp.groupby("Year"):
        year_rows.append({
            "Model": name,
            "Test Year": year,
            "RMSE": np.sqrt(group["Squared_Error"].mean()),
            "MAE": group["Absolute_Error"].mean(),
            "Observations": len(group)
        })

by_year_results = pd.DataFrame(year_rows)
display(by_year_results.round(5))
by_year_results.to_csv(DATA_DIR / "forecast_performance_by_year.csv", index=False)

## 13. Final forecasting comparison chart

In [ ]:
plot_results = overall_results[overall_results["Model"] != "Training-mean benchmark"].copy()
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(plot_results["Model"], plot_results["RMSE"])
axes[0].set_title("Out-of-Sample RMSE")
axes[0].set_ylabel("RMSE (lower is better)")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(plot_results["Model"], plot_results["MAE"])
axes[1].set_title("Out-of-Sample MAE")
axes[1].set_ylabel("MAE (lower is better)")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("Predictive Performance: Baseline vs F&O Models")
fig.tight_layout()
fig.savefig(GRAPH_DIR / "06_model_comparison.png", dpi=250)
plt.show()

## 14. Export summary tables and finish

The notebook has generated the regression summaries, forecast metrics, error percentiles, yearly results and six charts. The regression coefficient tables are generated directly from fitted models rather than typed in manually.

In [ ]:
# Save regression coefficient tables
def coefficient_table(fitted_model, filename):
    table = pd.DataFrame({
        "Coefficient": fitted_model.params,
        "Std_Error": fitted_model.bse,
        "Statistic": fitted_model.tvalues,
        "P_Value": fitted_model.pvalues,
        "CI_Lower": fitted_model.conf_int()[0],
        "CI_Upper": fitted_model.conf_int()[1]
    })
    table.to_csv(DATA_DIR / filename)
    return table

model_a_table = coefficient_table(model_a, "model_a_coefficients.csv")
model_b_table = coefficient_table(model_b, "model_b_coefficients.csv")

print("Model A: R-squared =", round(model_a.rsquared, 4),
      "| Observations =", int(model_a.nobs))
print("Model B: R-squared =", round(model_b.rsquared, 4),
      "| Observations =", int(model_b.nobs))
print("\nFiles created:")
for path in sorted(DATA_DIR.glob("*.csv")):
    print("-", path.name)
print("\nCharts created:")
for path in sorted(GRAPH_DIR.glob("*.png")):
    print("-", path.name)

## 15. Interpretation notes and limitations

- **Model A and Model B** explain historical 20-day rolling volatility. Model B includes lagged volatility, so its high R-squared largely reflects volatility persistence and must not be attributed to F&O variables alone.
- **Forecasting models** predict next-day absolute return. They are a separate exercise with a chronological holdout sample.
- In the project results, Market + F&O produced the lowest MAE (0.36116), while the recent-volatility baseline produced the lowest RMSE (0.52025). This is a mixed result: typical absolute error improved, but large errors remained.
- F&O activity and volatility may move together because of common market conditions, reverse causality or other omitted variables. The analysis does not establish causation.
- Results depend on the data source, classification decisions, study window and model specification.